In [38]:
#pip install langchain_huggingface
#!pip install langchain_community


In [39]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import  RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import spacy
import re
import contractions
from string import punctuation
from textblob import TextBlob
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

## 1. Load the document(.txt)

In [40]:
data= open('Data1.txt').read()

## 2. Text Normalization

## converting all character into lower case

In [41]:
data=data.lower()

## Removing extra spaces

In [42]:
data = re.sub(r'\s{2,}','',data)

# removing numbers like 1. , 2.

In [43]:
data= re.sub(r'\d+\.','',data)
data

'the quick brown fox jumps over the lazy dognatural language processing nlp is a fascinating field of artificial intelligence that focuses on enabling computersto understand interpret and generate human languagein a meaningful and useful way nlp combines computational linguistics—rulebased modeling of human language—withstatistical machine learning and deep learning models to processand analyze large amounts of natural language dataone of the most important tasks in nlp is text preprocessing which involves several steps tokenizationbreaking down text into individual words phrases or sentences for example dont worry becomes do nt worrylowercasingconverting all uppercase letters to lowercase for consistency hello world → hello worldremoving special characterspunctuationeliminating unnecessary symbols hello world → hello worldstopword removalfiltering out common words like the a is and or but in on at etcstemmingreducing words to their root form running runs ran → runlemmatizationconverti

## Contractions

In [44]:
#v= contractions.fix(data)

## Removing punctuations & Special characters

In [45]:
data= re.sub(r'[^0-9a-zA-Z\s]','',data)

## Handling Emoji's (use if data have emoji's)

## TextBlob

In [46]:
# values= TextBlob(data).correct()
# values

## Lemmatization

In [47]:
import spacy

In [48]:
nlp = spacy.load('en_core_web_sm')
tokens = nlp(data)

updated_tokens = [token.lemma_ for token in tokens if not token.is_stop ]

data = ' '.join(updated_tokens).strip()
data

'quick brown fox jump lazy dognatural language processing nlp fascinating field artificial intelligence focus enable computersto understand interpret generate human languagein meaningful useful way nlp combine computational linguisticsrulebase modeling human languagewithstatistical machine learning deep learning model processand analyze large amount natural language dataone important task nlp text preprocessing involve step tokenizationbreake text individual word phrase sentence example not worry not worrylowercasingconverte uppercase letter lowercase consistency hello world   hello worldremove special characterspunctuationeliminating unnecessary symbol hello world   hello worldstopword removalfiltering common word like etcstemmingreduce word root form run run run   runlemmatizationconverte word dictionarycanonical form well   good run   runmodern nlp application include machine translation google translate sentiment analysis analyze emotion review namedentity recognition neridentifye 

## Chunking( converting doc-> chunks)

In [49]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40
)
chunks = splitter.create_documents([data])
chunks


[Document(metadata={}, page_content='quick brown fox jump lazy dognatural language processing nlp fascinating field artificial intelligence focus enable computersto understand interpret generate human languagein meaningful useful way nlp'),
 Document(metadata={}, page_content='languagein meaningful useful way nlp combine computational linguisticsrulebase modeling human languagewithstatistical machine learning deep learning model processand analyze large amount natural'),
 Document(metadata={}, page_content='processand analyze large amount natural language dataone important task nlp text preprocessing involve step tokenizationbreake text individual word phrase sentence example not worry not'),
 Document(metadata={}, page_content='phrase sentence example not worry not worrylowercasingconverte uppercase letter lowercase consistency hello world   hello worldremove special characterspunctuationeliminating unnecessary symbol hello'),
 Document(metadata={}, page_content='unnecessary symbol he

In [50]:
#print(type(chunks[0]))
print(chunks[0].page_content)

quick brown fox jump lazy dognatural language processing nlp fascinating field artificial intelligence focus enable computersto understand interpret generate human languagein meaningful useful way nlp


In [51]:
print(chunks)

[Document(metadata={}, page_content='quick brown fox jump lazy dognatural language processing nlp fascinating field artificial intelligence focus enable computersto understand interpret generate human languagein meaningful useful way nlp'), Document(metadata={}, page_content='languagein meaningful useful way nlp combine computational linguisticsrulebase modeling human languagewithstatistical machine learning deep learning model processand analyze large amount natural'), Document(metadata={}, page_content='processand analyze large amount natural language dataone important task nlp text preprocessing involve step tokenizationbreake text individual word phrase sentence example not worry not'), Document(metadata={}, page_content='phrase sentence example not worry not worrylowercasingconverte uppercase letter lowercase consistency hello world   hello worldremove special characterspunctuationeliminating unnecessary symbol hello'), Document(metadata={}, page_content='unnecessary symbol hello 

In [52]:
chunks[0].metadata={'file_name':'data.txt'}
chunks

[Document(metadata={'file_name': 'data.txt'}, page_content='quick brown fox jump lazy dognatural language processing nlp fascinating field artificial intelligence focus enable computersto understand interpret generate human languagein meaningful useful way nlp'),
 Document(metadata={}, page_content='languagein meaningful useful way nlp combine computational linguisticsrulebase modeling human languagewithstatistical machine learning deep learning model processand analyze large amount natural'),
 Document(metadata={}, page_content='processand analyze large amount natural language dataone important task nlp text preprocessing involve step tokenizationbreake text individual word phrase sentence example not worry not'),
 Document(metadata={}, page_content='phrase sentence example not worry not worrylowercasingconverte uppercase letter lowercase consistency hello world   hello worldremove special characterspunctuationeliminating unnecessary symbol hello'),
 Document(metadata={}, page_content

In [53]:
#print(len(chunks))

In [54]:
print(chunks)

[Document(metadata={'file_name': 'data.txt'}, page_content='quick brown fox jump lazy dognatural language processing nlp fascinating field artificial intelligence focus enable computersto understand interpret generate human languagein meaningful useful way nlp'), Document(metadata={}, page_content='languagein meaningful useful way nlp combine computational linguisticsrulebase modeling human languagewithstatistical machine learning deep learning model processand analyze large amount natural'), Document(metadata={}, page_content='processand analyze large amount natural language dataone important task nlp text preprocessing involve step tokenizationbreake text individual word phrase sentence example not worry not'), Document(metadata={}, page_content='phrase sentence example not worry not worrylowercasingconverte uppercase letter lowercase consistency hello world   hello worldremove special characterspunctuationeliminating unnecessary symbol hello'), Document(metadata={}, page_content='un

### Embeddings( converts chunks to vectors)

In [55]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [56]:
vectordb= FAISS.from_documents(
    documents=chunks,
    embedding= embedding_model
)
vectordb

In [66]:
user_query = 'What is nlp?'
r_chunks = vectordb.similarity_search(user_query)

In [67]:
updated_r_chunks = set()

for chunk in r_chunks:
    updated_r_chunks.add(chunk.page_content)
R_text = '\n '.join(updated_r_chunks)
R_text

'word dictionarycanonical form well   good run   runmodern nlp application include machine translation google translate sentiment analysis analyze emotion review namedentity recognition neridentifye\n quick brown fox jump lazy dognatural language processing nlp fascinating field artificial intelligence focus enable computersto understand interpret generate human languagein meaningful useful way nlp\n processand analyze large amount natural language dataone important task nlp text preprocessing involve step tokenizationbreake text individual word phrase sentence example not worry not\n like alexa siri google assistant large language model llm like gpt4 bert t5and transformerbasedarchitecturesthe evolution nlp remarkable early approach rulebase system regular expressionsthen come'

In [ ]:
# def rag_query(query,k=3):
#     R_chunks = vectordb.similarity_search(query,k=k)
#     # print(R_chunks)
#     R_chunks = {doc.page_content for doc in R_chunks}
#     R_text = '\n'.join(R_chunks)
#     return R_text


# def g_text(rag_query,query):
#     import os
#     prompt = f''' 
#                  You're an helpful assistant
#                  Assigned Task for you : Structure my output =>{rag_query}
#                  for this input => {query}
#                  Note: 
#                  Don't add extra contents just structure mentioned output,
#                  If there is mistake in output correct or else keep the original output
#                  with strctured result.
#                  output structure:
#                  input:{query}
#                  output: Structured output
        
#         '''
        
        
#     llm_model  =ChatGoogleGenerativeAI(
#         model = "gemini-3.5-flash",  # gemini-3.6-flash,gemini-2.5-pro,gemini-2.5-flash
#         api_key = os.environ['GeminiAPI'],
#         temperature = 0.0,
        
#     )   
#     response = llm_model.invoke(user_prompt).content[0]['text']
#     return response   
    

# # print(response)
# user_prompt = 'Explain Machine Learning ?'
# user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)
# r_response =rag_query(user_prompt)
# g_response = g_text(r_response,user_prompt)
# print(g_response)

# # response = rag_query(user_prompt)
# # print(response)


### LCEL 1

## Step 1 Prompt Template

In [68]:
Structure_prompt_template = ChatPromptTemplate.from_messages(
    [
    ("system",
     "your are a helpful assistant."
     "check the retrived chunk score"
     "Example,Question => What is nlp ? Retrived out of 5 chunks 4 chunks are correct. Return the score:"
     "if any chunk is wrongly Retrived,Detect and Mention which Chunk is the Mistake and Explain why it is wrong."),
     ("human","Query : {user_query}"
              "Retrival Chunk : {r_chunks}")    
    ]
)

## step 2 LLM

In [ ]:
import os
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash", # gemini-3.6-flash,gemini-2.5-pro # os.enviorn
    api_key= os.environ["GEMINI_API_KEY"],
    temprature=0.0
)

Unexpected argument 'temprature' provided to ChatGoogleGenerativeAI. Did you mean: 'temperature'?
c:\anacondadistribution\Lib\site-packages\IPython\core\interactiveshell.py:3577: UserWarning: WARNING! temprature is not default parameter.
                temprature was transferred to model_kwargs.
                Please confirm that temprature is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)


## STEP 3 Parser

In [70]:
parser = StrOutputParser()

In [71]:
structure_chain= Structure_prompt_template | llm | parser

In [72]:
final_response= structure_chain.invoke({"user_query":user_query,"r_chunks":r_chunks})
final_response

'Based on the evaluation of the retrieved chunks for the query **"What is nlp?"**, here is the assessment:\n\n### **Score: 4 out of 4 chunks are correct (100%)**\n\n---\n\n### **Detailed Chunk Analysis:**\n\n1. **Chunk 1 (`46213d2d-d69b-4deb-b2cb-f56f38994dea`)** \n   * **Status:** **Correct**\n   * **Reason:** It directly defines NLP ("natural language processing nlp fascinating field artificial intelligence focus enable computers to understand interpret generate human language..."). This is the most direct answer to the query.\n\n2. **Chunk 2 (`f20d14c7-2d6a-4aed-a420-98c0d699fedf`)** \n   * **Status:** **Correct**\n   * **Reason:** It explains the processes and tasks involved in NLP (such as text preprocessing and tokenization), which helps explain *what* NLP does.\n\n3. **Chunk 3 (`7ca179fa-49d0-496a-b1b3-6d46922b3e68`)** \n   * **Status:** **Correct**\n   * **Reason:** It describes the evolution of NLP and lists famous applications/models (like Alexa, Siri, GPT-4, and BERT), provi

In [73]:
print(final_response)

Based on the evaluation of the retrieved chunks for the query **"What is nlp?"**, here is the assessment:

### **Score: 4 out of 4 chunks are correct (100%)**

---

### **Detailed Chunk Analysis:**

1. **Chunk 1 (`46213d2d-d69b-4deb-b2cb-f56f38994dea`)** 
   * **Status:** **Correct**
   * **Reason:** It directly defines NLP ("natural language processing nlp fascinating field artificial intelligence focus enable computers to understand interpret generate human language..."). This is the most direct answer to the query.

2. **Chunk 2 (`f20d14c7-2d6a-4aed-a420-98c0d699fedf`)** 
   * **Status:** **Correct**
   * **Reason:** It explains the processes and tasks involved in NLP (such as text preprocessing and tokenization), which helps explain *what* NLP does.

3. **Chunk 3 (`7ca179fa-49d0-496a-b1b3-6d46922b3e68`)** 
   * **Status:** **Correct**
   * **Reason:** It describes the evolution of NLP and lists famous applications/models (like Alexa, Siri, GPT-4, and BERT), providing context on wha